<a href="https://colab.research.google.com/github/IhorPetryshyn/Complex-Project-2/blob/main/Portfolio_Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Extracted data from the BIG Query database using SQL::

with session_info as (
SELECT
    date,
    sp.country,
    s.ga_session_id,
    sp.device,
    sp.continent,
    sp.channel,
    ab.test,
    ab.test_group,
from `DA.session_params` sp
join `DA.ab_test` ab
on sp.ga_session_id = ab.ga_session_id
join `DA.session` s
on s.ga_session_id = sp.ga_session_id
),


events as(
select
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    event_name,
    count (ep.ga_session_id) as session_with_events
from session_info
join `DA.event_params` ep
on ep.ga_session_id = session_info.ga_session_id
group by
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    event_name
),
session as(
select
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    count(distinct ga_session_id) as session_cnt
from session_info


group by
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
),


orders as(
select
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    count(distinct o.ga_session_id) as session_with_orders
from session_info
join `DA.order` o
on o.ga_session_id = session_info.ga_session_id
group by
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
),


accounts as(
select
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    count(distinct acs.ga_session_id) as new_account_cnt


from `DA.account_session` acs
join session_info
on session_info.ga_session_id = acs.ga_session_id
group by
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
)


Select    
    events.date,
    events.country,
    events.device,
    events.continent,
    events.channel,
    events.test,
    events.test_group,
    event_name,
    session_with_events as value,


from events


UNION ALL


Select    
    orders.date,
    orders.country,
    orders.device,
    orders.continent,
    orders.channel,
    orders.test,
    orders.test_group,
    'session with orders' as event_name,
    orders.session_with_orders as value,


from orders


UNION ALL


Select    
    accounts.date,
    accounts.country,
    accounts.device,
    accounts.continent,
    accounts.channel,
    accounts.test,
    accounts.test_group,
    'new account' as event_name,
    accounts.new_account_cnt as value
from accounts


UNION ALL


Select    
    session.date,
    session.country,
    session.device,
    session.continent,
    session.channel,
    session.test,
    session.test_group,
    'session' as event_name,
    session.session_cnt as value
from session


##Imported the generated CSV file into Google Colab and created an automated function for A/B testing


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
from statsmodels.stats.proportion import proportions_ztest
from google.colab import drive

drive.mount("/content/drive")
data = pd.read_csv("/content/drive/MyDrive/bq-results-20260608-091150-1780910015010/bq-results-20260608-083757-1780907914695.csv")
df = pd.DataFrame(data)
df["date"] = pd.to_datetime(df["date"])

metrics_definitions = {
    "add_payment_info / session": "add_payment_info",
    "add_shipping_info / session": "add_shipping_info",
    "begin_checkout / session": "begin_checkout",
    "new account / session": "new account"
}

def calculate_ab_significance(df_data, metrics_dict, alpha=0.05):
    results = []

    # Loop through each test for calculations
    for test_number, df_test_group in df_data.groupby('test'):

        # Calculate the number of sessions for control and test groups by filtering event_name as "session"
        sessions_A = df_test_group[(df_test_group['test_group'] == 1) & (df_test_group['event_name'] == 'session')]["value"].sum()
        sessions_B = df_test_group[(df_test_group['test_group'] == 2) & (df_test_group['event_name'] == 'session')]["value"].sum()

        # Using a loop so that if metrics change in the future, adjustments only need to be made in the dictionary
        # Calculate the number of conversions for each metric (numerator, while denominator remains the session count from above)
        for metric_name, event_name in metrics_dict.items():
            conversions_A = df_test_group[(df_test_group['test_group'] == 1) & (df_test_group['event_name'] == event_name')]["value"].sum()
            conversions_B = df_test_group[(df_test_group['test_group'] == 2) & (df_test_group['event_name'] == event_name')]["value"].sum()

            cr_A = conversions_A / sessions_A if sessions_A > 0 else 0
            cr_B = conversions_B / sessions_B if sessions_B > 0 else 0

            # Additional row to calculate the conversion lift of the test group compared to the control group
            lift = (cr_B - cr_A) / cr_A if cr_A > 0 else 0

            # Format data into arrays for statistical metric calculations
            count = np.array([conversions_A, conversions_B])
            nobs = np.array([sessions_A, sessions_B])

            # Calculate p_value and the statistical significance field (True or False):
            try:
                z_stat, p_value = proportions_ztest(count, nobs, alternative='two-sided')
                is_significant = p_value < alpha
            except:
                z_stat, p_value, is_significant = np.nan, np.nan, False

            # Append all values to the end of the results list:
            results.append({
                "test": test_number,
                "metric": metric_name,
                "conversions_A": conversions_A,
                "sessions_A": sessions_A,
                "cr_A": cr_A,
                "conversions_B": conversions_B,
                "sessions_B": sessions_B,
                "cr_B": cr_B,
                "lift": lift,
                "p_value": p_value,
                "is_significant": is_significant
            })
    return results

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   date        800996 non-null  datetime64[ns]
 1   country     800996 non-null  object        
 2   device      800996 non-null  object        
 3   continent   800996 non-null  object        
 4   channel     800996 non-null  object        
 5   test        800996 non-null  int64         
 6   test_group  800996 non-null  int64         
 7   event_name  800996 non-null  object        
 8   value       800996 non-null  int64         
dtypes: datetime64[ns](1), int64(3), object(5)
memory usage: 55.0+ MB


In [ ]:
final_results = []

total_res = calculate_ab_significance(df, metrics_definitions)
for row in total_res:
    row["segment_type"] = "all"
    row["segment_value"] = "all"
    final_results.append(row)

for country, group in df.groupby('country'):
    country_res = calculate_ab_significance(group, metrics_definitions)
    for i in country_res:
        i["segment_type"] = "country"
        i["segment_value"] = country
        final_results.append(i)

for device, group in df.groupby('device'):
    device_res = calculate_ab_significance(group, metrics_definitions)
    for i in device_res:
        i["segment_type"] = "device"
        i["segment_value"] = device
        final_results.append(i)

for channel, group in df.groupby('channel'):
    channel_res = calculate_ab_significance(group, metrics_definitions)
    for i in channel_res:
        i["segment_type"] = "channel"
        i["segment_value"] = channel
        final_results.append(i)

df_final = pd.DataFrame(final_results)
display(df_final)

df_final.to_csv("AB_test_results.csv", index=False)

/usr/local/lib/python3.12/dist-packages/statsmodels/stats/weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/proportion.py:1024: RuntimeWarning: invalid value encountered in sqrt
  std_diff = np.sqrt(var_)


,test,metric,conversions_A,sessions_A,cr_A,conversions_B,sessions_B,cr_B,lift,p_value,is_significant,segment_type,segment_value
0,1,add_payment_info / session,1988,45362,0.043825,2229,45193,0.049322,0.125420,0.000087,True,all,all
1,1,add_shipping_info / session,3034,45362,0.066884,3221,45193,0.071272,0.065605,0.009226,True,all,all
2,1,begin_checkout / session,3784,45362,0.083418,4021,45193,0.088974,0.066606,0.002894,True,all,all
3,1,new account / session,3823,45362,0.084278,3681,45193,0.081451,-0.033543,0.122859,False,all,all
4,2,add_payment_info / session,2344,50637,0.046290,2409,50244,0.047946,0.035769,0.214608,False,all,all
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1867,3,new account / session,383,4636,0.082614,363,4534,0.080062,-0.030897,0.654847,False,channel,Undefined
1868,4,add_payment_info / session,496,5716,0.086774,566,5862,0.096554,0.112708,0.068332,False,channel,Undefined
1869,4,add_shipping_info / session,640,5716,0.111966,666,5862,0.113613,0.014707,0.779457,False,channel,Undefined
1870,4,begin_checkout / session,1578,5716,0.276067,1600,5862,0.272944,-0.011312,0.706579,False,channel,Undefined


##CSV Файл з результатами коду:
https://drive.google.com/file/d/1RvWbN5xmTDkjfsW2tS0_8I7iXF2IZJ4g/view?usp=sharing

##Дешборд в Табло:
https://public.tableau.com/app/profile/ihor.petrsyhyn/viz/Portfolio2_17821301696490/Dashboard1?publish=yes


#**Conclusions & Business Recommendations**
#*Test 1:*   
**Segment: Device**

Desktop and Mobile: Demonstrate excellent growth across core metrics (shipping, payment, checkout). The mobile version for add_payment_info is particularly impressive, showing a lift of +17.14%.

Tablet: Shows a sharp drop (in red) across all first three metrics (e.g., -36.06% for payment and -33.20% for checkout). However, due to the very small sample size, fluctuations here are massive, and this drop is likely statistically insignificant. Nevertheless, it's worth checking as a hypothesis for developers: did the new layout break on tablet screens?

**Segment: Channel**

Direct: Excellent performance. Stable growth along the funnel (for instance, begin_checkout increased by +14.72%).

Organic Search: There is a hidden negative insight here. While the overall funnel suggests the test is successful, organic search shows a drop: -19.46% on payment, -9.44% on shipping, and -8.15% on checkout. Since organic search represents a significant share of the sample (~34.5%), this decline deserves a separate investigation. The new Variant B might be attracting less targeted traffic or users with different behaviors from search engines.

Undefined: Displays an abnormally high lift (over +50% across metrics). Since this channel's share is very small (~7.4%), this is also a result of high variance due to small data volumes.

##Business Recommendations for Test 1:
* Deploy Variant B for Desktop and Mobile. For the main platforms that generate nearly 98% of the traffic, the new design/algorithm is a clear winner.

* Isolate or highlight Organic Search for an additional test. Since there is a noticeable dip in organic traffic, this slice should be investigated in more detail.

* Check the technical status on Tablet. Even though the tablet traffic share is tiny, continuous red indicators are a classic marker of a technical bug (e.g., the payment button overlapping another screen element).

#*Test 2:*
**Segment: Device**

Desktop: Shows a slight visual increase across all major funnel metrics (ranging from +5.45% to +8.00%). Since desktop accounts for the lion's share of traffic (58.47%), it slightly pulled the overall numbers into a small positive zone. However, considering the overall neutral picture, even this growth is not enough to draw confident conclusions.

Tablet: Once again, we see a heavy drop in checkout (-21.53%). But taking the session distribution into account (2.21%), there is simply not enough data on tablets. These charts cannot be trusted due to extreme variance.

**Segment: Channel**

Organic Search: The trend from the previous test repeats itself—a steady decline across the entire funnel for Variant B (e.g., -14.28% on payment and -9.48% on checkout). Since this segment is large (~34.7%), this is a consistent pattern. The new UI changes in Variant B systematically displease users coming from search engines.

Social Search and Paid Search: Show visual growth (for example, Social Media yielded +18.17% at the payment stage). However, their traffic shares are too small to outweigh the overall neutral result.

## Business Recommendations for Test 2:
* The test is concluded as neutral (Unsuccessful). The changes in Variant B did not bring any profit to the business. Implementing Variant B for the general public in its current form is not recommended. It is advised to roll back 100% of the traffic to Variant A.

* Investigate search engine traffic behavior. For two tests in a row, Variant B hurts the conversion of users coming from search engines. It is worth conducting a qualitative study (e.g., UX testing): perhaps search users look for specific information or items, and the new design elements of Variant B prevent them from reaching their goal quickly.

#*Test 3:*
**Segment: Device**

Desktop: Demonstrates relative stability, though the begin_checkout stage dropped by -1.71%.

Mobile: This is the primary source of the decline! On mobile devices, begin_checkout dropped by -4.70%, and adding shipping info (add_shipping_info) fell by -2.10%. Since mobile traffic is huge (38.93%), this drop significantly impacted the overall numbers.

Tablet: Traditionally displays large red columns (drops up to -26.31%), but again, we note the tiny sample share (2.23%)—variance is high here.

**Segment: Channel**

Organic Search: As seen in previous test versions, organic traffic (which takes up ~35.5% of all traffic) continues to consistently turn "red". begin_checkout sank by -8.02%, and shipping dropped by -4.20%.

Direct and Paid Search: Also show a negative trend at key stages of the funnel (for instance, Direct dropped by -7.72% at the payment stage).

Undefined: Shows a strong blue increase, but due to the low channel share, it does not save the overall situation.

##Business Recommendations for Test 3:
* Reject Variant B and keep Variant A. The innovations proved unsuccessful: they significantly reduce conversion to checkout (especially on mobile devices and in organic search). Rolling out this variant to 100% of users is not acceptable, as the business will lose money.

* UX/UI audit of the mobile version: Since mobile users experienced a heavy drop at the start of the checkout stage, it is necessary to check exactly how the new element renders on smartphones.

#*Test 4:*
**Segment: Device**

Desktop: This is the main reason for the decline in the entire test. On computers (which make up 58.43% of all traffic), we see solid red columns across the entire funnel. Payment dropped by -7.39%, shipping by -6.40%, and checkout by -5.69%.

Mobile: Unlike desktop, mobile users reacted to Variant B in a neutral-to-positive way (checkout grew by +4.29%, payment by +5.90%).

Analytical Insight: The changes in Variant B were clearly designed without considering the desktop interface. What looks fine on mobile phones seems to have broken the familiar user journey on large PC screens.

**Segment: Channel**

Organic Search: As with all your tests, the large organic segment (~36% of traffic) again demonstrates a steady decline in Variant B (begin_checkout fell by -7.96%). This confirms the overall global negative outcome of the test.

Social Search: A major anomaly is observed here — adding payment info dropped by a massive -20.78%. Although this channel's share is small (~7.6%), such a deep negative result combined with the desktop drop is an alarming signal.

## Business Recommendations for Test 4
* Strongly reject Variant B. This interface or logic variant brings financial losses to the business, significantly reducing the number of registrations and checkout transitions. It is required to roll back 100% of the traffic to Variant A.

* Analyze the desktop version of Variant B. Since Desktop was precisely what dragged the entire test down, designers and developers should review the site layout on monitors. There might be unnecessary elements that distract users from registering and purchasing.